In [12]:
import os
import pickle
import pandas as pd

def find_trafo_pickle_paths():
    cwd = os.getcwd()
    parent = os.path.dirname(cwd)
    candidates = [
        cwd, parent,
        os.path.join(parent, "results"),
        os.path.join(parent, "results", "metrics"),
        os.path.join(parent, "notebooks"),
        "/content",
    ]
    paths = []
    for root in candidates:
        if not os.path.isdir(root):
            continue
        try:
            for f in os.listdir(root):
                if f.startswith("results_TRAFO_") and f.endswith(".pkl"):
                    paths.append(os.path.join(root, f))
        except OSError:
            pass
    return paths

paths = find_trafo_pickle_paths()
if not paths and os.path.isdir("/content"):
    try:
        paths = [os.path.join("/content", f) for f in os.listdir("/content") if f.startswith("results_TRAFO_") and f.endswith(".pkl")]
    except OSError:
        pass
print("Archivos encontrados:", paths)

Archivos encontrados: ['/Users/guane/Documentos/Doctorate/MSRRFF-Wind-Forecast/results/metrics/results_TRAFO_Synthetic.pkl', '/Users/guane/Documentos/Doctorate/MSRRFF-Wind-Forecast/results/metrics/results_TRAFO_Argone.pkl', '/Users/guane/Documentos/Doctorate/MSRRFF-Wind-Forecast/results/metrics/results_TRAFO_Netherland-1.pkl', '/Users/guane/Documentos/Doctorate/MSRRFF-Wind-Forecast/results/metrics/results_TRAFO_Netherland-0.pkl', '/Users/guane/Documentos/Doctorate/MSRRFF-Wind-Forecast/results/metrics/results_TRAFO_Netherland-2.pkl', '/Users/guane/Documentos/Doctorate/MSRRFF-Wind-Forecast/results/metrics/results_TRAFO_Chengdu.pkl', '/Users/guane/Documentos/Doctorate/MSRRFF-Wind-Forecast/results/metrics/results_TRAFO_Beijing.pkl']


In [13]:
COL_ORDER = [
    "Dataset", "Model", "horizon",
    "lr", "epochs", "d_model", "nhead", "dropout",
    "num_layers_enc", "num_layers_dec",
    "num_layers", "kernel_size", "n_modes",
]

rows = []
for p in paths:
    with open(p, "rb") as f:
        exp = pickle.load(f)
    dataset = exp.get("config", {}).get("folder_name", os.path.splitext(os.path.basename(p))[0].replace("results_TRAFO_", ""))
    for model_name, info in exp.get("models", {}).items():
        args = info.get("args", {})
        args.pop("model_name", None)
        row = {"Dataset": dataset, "Model": model_name}
        for k, v in args.items():
            row[k] = v
        rows.append(row)

df = pd.DataFrame(rows)
df = df[[c for c in COL_ORDER if c in df.columns]]
for c in COL_ORDER:
    if c not in df.columns:
        df[c] = pd.NA
df = df[COL_ORDER]
df["lr"] = df["lr"].round(6)
df["dropout"] = df["dropout"].round(4)
df = df.sort_values(["Dataset", "Model"]).reset_index(drop=True)
df

,Dataset,Model,horizon,lr,epochs,d_model,nhead,dropout,num_layers_enc,num_layers_dec,num_layers,kernel_size,n_modes
0,Argone,Autoformer,7,0.001383,28,96,8,0.1313,NaN,NaN,2.0,13.0,NaN
1,Argone,FEDformer,7,0.000136,54,256,8,0.1467,NaN,NaN,1.0,NaN,25.0
2,Argone,Transformer,7,0.000357,58,64,1,0.1031,4.0,2.0,NaN,NaN,NaN
3,Beijing,Autoformer,7,0.000291,40,64,8,0.0966,NaN,NaN,3.0,25.0,NaN
4,Beijing,FEDformer,7,0.000141,47,192,2,0.2623,NaN,NaN,3.0,NaN,24.0
5,Beijing,Transformer,7,0.000357,58,64,1,0.1031,4.0,2.0,NaN,NaN,NaN
6,Chengdu,Autoformer,7,0.002082,42,96,4,0.1344,NaN,NaN,2.0,13.0,NaN
7,Chengdu,FEDformer,7,0.002196,29,128,1,0.1839,NaN,NaN,2.0,NaN,23.0
8,Chengdu,Transformer,7,0.000186,27,256,8,0.1786,4.0,1.0,NaN,NaN,NaN
9,Netherland-0,Autoformer,10,0.001196,46,64,2,0.2091,NaN,NaN,2.0,25.0,NaN


In [14]:
out_dir = os.path.join(os.path.dirname(os.getcwd()), "results", "plots")
if os.path.basename(os.getcwd()) != "notebooks":
    out_dir = os.path.join(os.getcwd(), "results", "plots")
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "transformer_hyperparameters.csv")
df.to_csv(out_path, index=False)
print("Guardado:", out_path)

Guardado: /Users/guane/Documentos/Doctorate/MSRRFF-Wind-Forecast/results/plots/transformer_hyperparameters.csv


In [ ]:
def find_plain_rnn_pickle_paths():
    cwd = os.getcwd()
    parent = os.path.dirname(cwd)
    candidates = [
        cwd, parent,
        os.path.join(parent, "results"),
        os.path.join(parent, "results", "metrics"),
        os.path.join(parent, "notebooks"),
        "/content",
    ]
    paths = []
    for root in candidates:
        if not os.path.isdir(root):
            continue
        try:
            for f in os.listdir(root):
                if f.startswith("results_") and not f.startswith("results_TRAFO_") and f.endswith(".pkl"):
                    paths.append(os.path.join(root, f))
        except OSError:
            pass
    return paths

paths_plain = find_plain_rnn_pickle_paths()
if not paths_plain and os.path.isdir("/content"):
    try:
        paths_plain = [os.path.join("/content", f) for f in os.listdir("/content") if f.startswith("results_") and not f.startswith("results_TRAFO_") and f.endswith(".pkl")]
    except OSError:
        pass

COL_PLAIN = ["Dataset", "Model", "horizon", "hidden", "num_layers"]
rows_plain = []
for p in paths_plain:
    with open(p, "rb") as f:
        exp = pickle.load(f)
    dataset = exp.get("config", {}).get("folder_name", os.path.splitext(os.path.basename(p))[0].replace("results_", ""))
    for model_name, info in exp.get("models", {}).items():
        if info.get("kind") != "plain":
            continue
        args = info.get("args", {})
        row = {"Dataset": dataset, "Model": args.get("rnn_type", model_name)}
        for k in ["horizon", "hidden", "num_layers"]:
            row[k] = args.get(k)
        rows_plain.append(row)

df_plain = pd.DataFrame(rows_plain)
for c in COL_PLAIN:
    if c not in df_plain.columns:
        df_plain[c] = pd.NA
df_plain = df_plain[COL_PLAIN]
if not df_plain.empty:
    df_plain = df_plain.sort_values(["Dataset", "Model"]).reset_index(drop=True)
df_plain

,Dataset,Model,horizon,hidden,num_layers
0,Argone,GRU,7,80,2
1,Argone,LSTM,7,128,3
2,Argone,RNN,7,96,3
3,Beijing,GRU,7,80,2
4,Beijing,LSTM,7,128,3
5,Beijing,RNN,7,160,3
6,Chengdu,GRU,7,224,1
7,Chengdu,LSTM,7,128,3
8,Chengdu,RNN,7,96,3
9,Netherland-0,GRU,10,80,2


In [16]:
out_dir = os.path.join(os.path.dirname(os.getcwd()), "results", "plots")
if os.path.basename(os.getcwd()) != "notebooks":
    out_dir = os.path.join(os.getcwd(), "results", "plots")
os.makedirs(out_dir, exist_ok=True)
out_path_plain = os.path.join(out_dir, "plain_rnn_hyperparameters.csv")
df_plain.to_csv(out_path_plain, index=False)
print("Guardado:", out_path_plain)

Guardado: /Users/guane/Documentos/Doctorate/MSRRFF-Wind-Forecast/results/plots/plain_rnn_hyperparameters.csv


In [17]:
paths_rff = find_plain_rnn_pickle_paths()
rows_rff = []
for p in paths_rff:
    with open(p, "rb") as f:
        exp = pickle.load(f)
    dataset = exp.get("config", {}).get("folder_name", os.path.splitext(os.path.basename(p))[0].replace("results_", ""))
    for model_name, info in exp.get("models", {}).items():
        if info.get("kind") != "rff":
            continue
        args = info.get("args", {})
        rnn_type = args.get("rnn_type", model_name.replace("RFF_", "") if isinstance(model_name, str) else model_name)
        row = {"Dataset": dataset, "Model": rnn_type}
        for k in ["horizon", "hidden", "num_layers", "bands_preset", "nf_per_band", "spectral_dropout_p", "lr_rho_multiplier"]:
            row[k] = args.get(k)
        rows_rff.append(row)

COL_RFF = ["Dataset", "Model", "horizon", "hidden", "num_layers", "bands_preset", "nf_per_band", "spectral_dropout_p", "lr_rho_multiplier"]
df_rff = pd.DataFrame(rows_rff)
for c in COL_RFF:
    if c not in df_rff.columns:
        df_rff[c] = pd.NA
df_rff = df_rff[COL_RFF]
if not df_rff.empty:
    df_rff["spectral_dropout_p"] = pd.to_numeric(df_rff["spectral_dropout_p"], errors="coerce").round(4)
    df_rff["lr_rho_multiplier"] = pd.to_numeric(df_rff["lr_rho_multiplier"], errors="coerce").round(4)
    df_rff = df_rff.sort_values(["Dataset", "Model"]).reset_index(drop=True)
df_rff

,Dataset,Model,horizon,hidden,num_layers,bands_preset,nf_per_band,spectral_dropout_p,lr_rho_multiplier
0,Argone,GRU,7,256,3,None,24,0.1,NaN
1,Argone,LSTM,7,96,3,None,48,0.3,NaN
2,Argone,RNN,7,32,3,None,48,0.2,NaN
3,Beijing,GRU,7,256,3,None,24,0.1,NaN
4,Beijing,LSTM,7,144,3,None,104,0.2,NaN
5,Beijing,RNN,7,32,3,None,48,0.2,NaN
6,Chengdu,GRU,7,256,3,None,24,0.1,NaN
7,Chengdu,LSTM,7,96,3,None,48,0.3,NaN
8,Chengdu,RNN,7,48,1,None,32,0.5,NaN
9,Netherland-0,GRU,10,80,3,None,56,0.2,NaN


In [18]:
out_dir = os.path.join(os.path.dirname(os.getcwd()), "results", "plots")
if os.path.basename(os.getcwd()) != "notebooks":
    out_dir = os.path.join(os.getcwd(), "results", "plots")
os.makedirs(out_dir, exist_ok=True)
out_path_rff = os.path.join(out_dir, "rff_rnn_hyperparameters.csv")
df_rff.to_csv(out_path_rff, index=False)
print("Guardado:", out_path_rff)

Guardado: /Users/guane/Documentos/Doctorate/MSRRFF-Wind-Forecast/results/plots/rff_rnn_hyperparameters.csv
